# Movie KV pregen (ReasonDB text server, Qwen2.5-0.5B)

Runs [`run_movie_kv_pregen.py`](run_movie_kv_pregen.py) → **`KvTextQaModelWrapper.prepare_caches`** in [`kv_cache_text_qa_server_new.py`](kv_cache_text_qa_server_new.py).

**Not used:** `run_cache_generation.py`, `cache_generation_manager.py`, `kv_cache_image_qa_server.py`.

## Workflow

1. **GPU** + **Internet** + **`HF_TOKEN`** secret.
2. **Commit and push** this repo to GitHub (`main` must contain `benchmarks/kv_cache_pregen/run_movie_kv_pregen.py` and deps).
3. **Add Data** only for `reviews_1000.csv` (no code zip).
4. Notebook **shallow-clones** `BENCH_URL` into `/kaggle/working/kv-compression-benchmark`.
5. Set **`RUN_PRESS`** per session: `ea` | `kvzip` | `finch_no_cpt` | `finch_with_cpt`.
6. Smoke (`SMOKE_MODE=True`, no checkpoint) then full run (`SMOKE_MODE=False`).
7. Download **Output** before the session ends.

Output: `{CACHE_DIR}/{model}/{press_name}/comp{ratio}/cache_entry_{sha256}.pt`

In [ ]:
!pip install -q "kvpress>=0.2" "transformers>=4.45" accelerate pandas tqdm safetensors sentencepiece protobuf pyyaml

In [ ]:
from __future__ import annotations

import os
import shutil
import sys
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient

    _sec = UserSecretsClient()
    for name in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        try:
            t = _sec.get_secret(name)
            if t:
                os.environ["HF_TOKEN"] = t
                os.environ["HUGGING_FACE_HUB_TOKEN"] = t
                print("Loaded secret:", name)
                break
        except Exception:
            continue
except Exception as e:
    print("Secrets:", e)

CLONE_REPO = True
BENCH_URL = "https://github.com/physical168/kv-compression-benchmark.git"
GIT_BRANCH: str | None = None  # e.g. "main"
WORK_ROOT = Path("/kaggle/working")
REPO_ROOT = WORK_ROOT / "kv-compression-benchmark"
WORK_BUNDLE = REPO_ROOT / "benchmarks/kv_cache_pregen"

base = Path("/kaggle/working/movie_kv_pregen_qwen05b")
CACHE_DIR = base / "caches"
CHECKPOINT_CSV = base / "movie_kv_pregen_checkpoint.csv"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_CSV.parent.mkdir(parents=True, exist_ok=True)

os.environ["MOVIE_KV_PREGEN"] = "1"  # batch pregen: skip Flask imports
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

RUN_PRESS = "ea"  # ea | kvzip | finch_no_cpt | finch_with_cpt
_allowed = {"ea", "kvzip", "finch_no_cpt", "finch_with_cpt"}
if RUN_PRESS not in _allowed:
    raise ValueError(f"RUN_PRESS must be one of {_allowed}")

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
TAIL = 500
RATIOS = ["0.2", "0.4", "0.6", "0.8"]

SMOKE_MODE = False
SMOKE_MAX_ROWS = 2
SMOKE_RATIOS = ["0.2"]

print("RUN_PRESS:", RUN_PRESS)
print("CLONE_REPO:", CLONE_REPO, "BENCH_URL:", BENCH_URL)
print("WORK_BUNDLE:", WORK_BUNDLE)
print("CACHE_DIR:", CACHE_DIR.resolve())

In [ ]:
from __future__ import annotations

import shutil
import subprocess
from pathlib import Path

REQUIRED = [
    "run_movie_kv_pregen.py",
    "kv_cache_text_qa_server_new.py",
    "text_kvpress_patch.py",
    "reasondb/backends/kv_cache_base.py",
    "queries_workloads/_workload.yaml",
]
MARKER = Path("benchmarks/kv_cache_pregen/run_movie_kv_pregen.py")


def _bundle_complete(root: Path) -> bool:
    bundle = root / "benchmarks/kv_cache_pregen"
    return bundle.is_dir() and all((bundle / r).is_file() for r in REQUIRED)


def find_reviews_csv() -> Path:
    for p in Path("/kaggle/input").rglob("reviews_1000.csv"):
        return p
    raise FileNotFoundError("Add Data: reviews_1000.csv")


def find_repo_root() -> Path:
    if CLONE_REPO:
        dest = REPO_ROOT
        if _bundle_complete(dest):
            return dest
        if dest.exists():
            shutil.rmtree(dest)
        clone_cmd = ["git", "clone", "--depth", "1"]
        if GIT_BRANCH:
            clone_cmd += ["--branch", GIT_BRANCH]
        clone_cmd += [BENCH_URL, str(dest)]
        subprocess.check_call(clone_cmd)
        if not _bundle_complete(dest):
            missing = [r for r in REQUIRED if not (dest / "benchmarks/kv_cache_pregen" / r).is_file()]
            raise FileNotFoundError(
                f"After git clone, missing under {dest / 'benchmarks/kv_cache_pregen'}: {missing}\n"
                f"Push latest code to {BENCH_URL}, then re-run this cell."
            )
        return dest
    for p in Path("/kaggle/input").rglob("run_movie_kv_pregen.py"):
        root = p.parents[2] if len(p.parents) >= 3 else p.parent
        if _bundle_complete(root):
            return root
    raise FileNotFoundError(
        "Set CLONE_REPO=True with Internet, or add a dataset containing the full repo."
    )


REPO_ROOT = find_repo_root()
WORK_BUNDLE = REPO_ROOT / "benchmarks/kv_cache_pregen"
sys.path.insert(0, str(WORK_BUNDLE))

CSV_PATH = find_reviews_csv()
SCRIPT = WORK_BUNDLE / "run_movie_kv_pregen.py"

for r in REQUIRED:
    assert (WORK_BUNDLE / r).is_file(), f"Missing {r}"

print("REPO_ROOT:", REPO_ROOT)
print("WORK_BUNDLE:", WORK_BUNDLE)
print("CSV:", CSV_PATH)
print("SCRIPT:", SCRIPT)

## Smoke test

Set `SMOKE_MODE=True` in the config cell. **No `--checkpoint-csv`** on smoke runs.

In [ ]:
import importlib.util
import shlex
import subprocess

import torch

if not SMOKE_MODE:
    print("SMOKE_MODE is False - skipping smoke.")
else:
    cmd = [
        sys.executable,
        str(SCRIPT),
        "--csv",
        str(CSV_PATH),
        "--cache-dir",
        str(CACHE_DIR),
        "--model",
        MODEL,
        "--tail",
        str(int(TAIL)),
        "--max-rows",
        str(int(SMOKE_MAX_ROWS)),
        "--press-name",
        RUN_PRESS,
        "--compression-ratio",
        *SMOKE_RATIOS,
        "--cpt-csv",
        str(CSV_PATH),
    ]
    if RUN_PRESS in ("ea", "kvzip"):
        cmd.append("--kvzip-patch")

    print("Smoke:", shlex.join(cmd))
    subprocess.check_call(cmd, cwd=str(WORK_BUNDLE))

    from kv_cache_text_qa_server_new import KvTextQaModelWrapper

    press = {
        "ea": "expected_attention",
        "kvzip": "kvzip",
        "finch_no_cpt": "finch",
        "finch_with_cpt": "finch-cachenotes",
    }[RUN_PRESS]
    tag = SMOKE_RATIOS[0].replace(".", "_")
    out_dir = CACHE_DIR / MODEL / press / f"comp{tag}"
    pts = list(out_dir.glob("cache_entry_*.pt"))
    assert pts, f"No .pt under {out_dir}"
    try:
        obj = torch.load(pts[0], map_location="cpu", weights_only=False)
    except TypeError:
        obj = torch.load(pts[0], map_location="cpu")
    print("Smoke OK:", pts[0].name, type(obj))

## Full run

Set `SMOKE_MODE=False`. Uses checkpoint CSV for `(press_name, compression_ratio)` resume.

In [ ]:
import shlex
import subprocess

if SMOKE_MODE:
    raise RuntimeError("Set SMOKE_MODE=False before the full run.")

cmd = [
    sys.executable,
    str(SCRIPT),
    "--csv",
    str(CSV_PATH),
    "--cache-dir",
    str(CACHE_DIR),
    "--model",
    MODEL,
    "--tail",
    str(int(TAIL)),
    "--press-name",
    RUN_PRESS,
    "--compression-ratio",
    *RATIOS,
    "--cpt-csv",
    str(CSV_PATH),
    "--checkpoint-csv",
    str(CHECKPOINT_CSV),
]
if RUN_PRESS in ("ea", "kvzip"):
    cmd.append("--kvzip-patch")

print("Full:", shlex.join(cmd))
subprocess.check_call(cmd, cwd=str(WORK_BUNDLE))

## Optional: zip Output

In [ ]:
# import shutil
# shutil.make_archive("/kaggle/working/movie_kv_qwen05b", "zip", root_dir=str(base))
# print("Wrote /kaggle/working/movie_kv_qwen05b.zip")